# 30. UI 개선 학습 노트 — 예시 질문 버튼 · 대화 지우기 (2026-11-09)

이 노트북은 오늘 `src/rag/ui/app.py` 에 넣은 두 기능의 **완성 코드**를 주석과 함께 정리한 학습 자료다.
실제 코드는 `src/rag/ui/app.py`, 시험은 `tests/rag/ui/test_app_buttons.py` 에 있고, 여기 코드는 같은 내용이다.

## 이 노트에서 배우는 것
1. Streamlit은 **버튼을 누를 때마다 스크립트 전체를 처음부터 다시 실행**한다(재실행 모델).
2. 버튼이 값을 넘기려면 `on_click` 콜백 + `st.session_state` 를 쓴다.
3. 한 번만 써야 하는 값은 `session_state.pop(키, None)` 으로 꺼낸다.
4. 화면 흐름은 `AppTest` 로 DB·LLM 없이 시험할 수 있다.

## 원복 방법 (변경 전 상태로 되돌리기)
| 방법 | 명령 | 설명 |
|---|---|---|
| 로컬 태그 | `git checkout pre-ui-20261109 -- src/rag/ui/app.py` | 변경 전 커밋(1cb6444)의 app.py 로 되돌린다. 로컬 태그이며 원격에는 올리지 않았다 |
| 백업 파일 | `notebooks/prevention_legal/backups/app_before_ui_20261109.py` 를 app.py 에 덮어쓴다 | Git 제외 폴더라 이 컴퓨터에만 있다 |
| 새 시험 파일 | `tests/rag/ui/test_app_buttons.py` 삭제 | 기능을 되돌리면 이 시험은 실패하므로 함께 뺀다 |

> 원복해도 `docs/progress.md` 의 오늘 기록은 남겨 두고, 원복 사실을 한 줄 덧붙인다(과거 기록은 지우지 않는다).

## 1. 재실행 모델 먼저 이해하기
Streamlit 화면의 `st.button("...")` 은 **클릭된 직후의 한 번의 실행에서만 `True`** 를 돌려주고, 바로 다음 실행에서는 다시 `False` 가 된다.
그래서 "버튼을 눌렀을 때 질문을 실행"하려고 `if st.button(...): 실행()` 으로 쓰면 이미 그려 놓은 화면 아래에서 실행되고,
이어지는 재실행에서 흔적이 사라지는 문제가 생긴다.

이 프로젝트의 질문 실행 경로(`main()`)는 이렇게 흐른다.
```
chat_input 으로 질문 입력 → answer_live(질문 실행·스트리밍) → history 에 결과 추가 → st.rerun() → 최종 결과로 다시 그림
```
예시 버튼도 **같은 경로**를 타게 해야 한다. 버튼이 직접 질문을 실행하면 같은 질문이 두 번 실행되거나 기록이 중복될 수 있다.
(Codex 검토에서도 이 점을 가장 큰 위험으로 지적했다.)

해결 방법: 버튼은 "질문을 맡겨 두기"만 하고, 실행은 기존 경로가 한다.

In [ ]:
# ── 설정값 (app.py 상단) ─────────────────────────────────────────────
# 예시 질문은 11/08 메인 화면에서 실제로 실행해 확인한 것만 둔다.
#   Q10 = ANSWERED(법 18·19쪽), Q11 = ANSWERED(법 19쪽), Q23 = PARTIAL(메인 라우팅 한계로 법 인용 없음)
# 질문 문구는 바꾸지 않는다. 문구가 달라지면 11/08 확인 결과와 비교할 수 없게 된다.
# (버튼에 보일 짧은 이름, 실제로 실행할 질문) 쌍의 목록이다.
EXAMPLE_QUESTIONS = [
    ("의사 신고 vs 학교장 신고", "의사가 하는 신고와 학교장 등이 하는 신고는 법적으로 어떻게 구분되나요?"),
    ("신고의무자가 아닌 사람이 발견하면", "신고의무자가 아닌 사람이 감염병 환자로 의심되는 사람을 발견하면 어떻게 해야 하나요?"),
    ("백일해 접촉자 안내", "학교에서 백일해 환자가 발생했을 때 접촉자 안내는 어떻게 하나요?"),
]

# 예시 버튼이 누른 질문을 "다음 재실행"으로 넘기는 session_state 키.
# 키 이름을 상수로 한 곳에 두면 오타로 다른 키(history, saved, polish, allowed 등)와 어긋나거나 겹치는 일을 막는다.
PENDING_KEY = "pending_question"

## 2. 콜백 함수 3개
`on_click=함수` 로 연결한 콜백은 **다음 재실행이 시작되기 전에** 먼저 실행된다.
그래서 콜백에서 `session_state` 를 바꿔 두면, 재실행된 `main()` 이 바뀐 값을 그대로 읽는다.

In [ ]:
try:
    import streamlit as st
except ImportError:
    # streamlit 이 없는 노트북 커널에서도 아래 콜백 로직을 따라 해 볼 수 있게 session_state 만 흉내 낸다.
    from types import SimpleNamespace
    st = SimpleNamespace(session_state={})


def pick_example(text: str) -> None:
    """예시 버튼의 on_click 콜백. 질문을 맡겨 두기만 한다.
    여기서 질문을 직접 실행하지 않는 이유: 실행·기록·화면 갱신은 main() 의 한 경로(채팅 입력과 동일)가 맡아야
    중복 실행·중복 기록을 피할 수 있기 때문이다."""
    st.session_state[PENDING_KEY] = text


def clear_conversation() -> None:
    """'현재 화면 대화 지우기' 버튼의 on_click 콜백.
    지우는 것은 화면 대화(history) 하나뿐이다. 아래 세 가지는 일부러 건드리지 않는다.
      - 저장한 답변(saved): 사용자가 📌로 저장한 것이므로 별도 삭제 버튼으로만 지운다.
      - 인증·개발자 설정: 세션 전체를 지우면 비밀번호 입력 상태가 풀리거나 개발자 설정이 초기화될 수 있다.
      - 서버의 질문 이력 파일: 서비스 점검용 로그라 화면 버튼으로 지워지지 않는다(개인정보 지움으로 오해하지 않게 버튼 문구에 적었다)."""
    st.session_state["history"] = []


def show_examples() -> None:
    """대화가 비어 있을 때 예시 질문 버튼을 그린다.
    key 는 버튼마다 달라야 한다(같으면 Streamlit이 중복 위젯 오류를 낸다).
    help 로 마우스를 올리면 실제 질문 전체가 보이게 했다(라벨은 짧게 줄였기 때문)."""
    st.caption("예시 질문을 눌러 바로 물어볼 수 있습니다.")
    for index, (label, text) in enumerate(EXAMPLE_QUESTIONS):
        st.button(f"💬 {label}", key=f"example-{index}", on_click=pick_example, args=(text,), help=text)

## 3. `main()` 에 끼워 넣은 부분
핵심은 세 줄이다. **원래 있던 코드(`question = st.chat_input(...)`)는 변수 이름만 `typed` 로 바꿨다.**

In [ ]:
# ── main() 안 (질문 받는 부분) ───────────────────────────────────────
#
# 변경 전:  question = st.chat_input("질문을 입력하세요 ...")
# 변경 후:
# typed = st.chat_input("질문을 입력하세요 (예: 결핵 확진 학생이 나오면 담임교사는 무엇을 하나요?)")

# pop(키, None): 값을 꺼내면서 session_state 에서 지운다. 값이 없으면 None.
# → 예시 버튼이 맡긴 질문은 이번 재실행에서 한 번만 쓰이고, 그다음 재실행에서는 없다.
#   (get 으로 읽기만 하면 질문이 남아서, 아무것도 안 눌러도 같은 질문이 계속 실행될 수 있다.)
# pending = st.session_state.pop(PENDING_KEY, None)

# 직접 입력한 질문이 있으면 그것이 우선이다. `typed or pending`: typed 가 None/빈 문자열이면 pending 을 쓴다.
# question = typed or pending
# 이후 코드(if question: ... answer_live ... history 추가 ... st.rerun())는 전혀 바꾸지 않았다.

# ── "질문하기" 탭 안 (history 를 그리기 전) ─────────────────────────
#
# if not history_items:
#     show_examples()              # 대화가 비어 있으면 예시 버튼
# else:
#     st.button("🧹 현재 화면 대화 지우기", key="clear-conversation", on_click=clear_conversation,
#               help="화면에 쌓인 대화만 지웁니다. '저장한 답변'과 서비스 점검용 질문 기록은 그대로 남습니다.")

## 4. 흐름 한눈에 보기
```
[예시 버튼 클릭]
   └ on_click: pick_example → session_state["pending_question"] = 질문
        └ 스크립트 재실행: main()
             ├ typed   = chat_input()      → None (입력 없음)
             ├ pending = pop("pending_question") → 질문 (그리고 키는 지워짐)
             ├ question = typed or pending  → 질문
             └ if question: answer_live → history 추가 → st.rerun()   ← 채팅 입력과 같은 경로
                  └ 재실행: pending 은 이미 없음 → 같은 질문이 다시 실행되지 않음

[대화 지우기 클릭]
   └ on_click: clear_conversation → session_state["history"] = []
        └ 재실행: history_items 가 비어 있음 → 예시 버튼이 다시 보임 (saved 는 그대로)
```

## 5. 시험으로 확인한 것 (`tests/rag/ui/test_app_buttons.py`)
| 시험 | 확인하는 약속 |
|---|---|
| 빈 화면 | 예시 버튼 3개가 보이고 지우기 버튼은 없다 |
| 예시 클릭 | 질문이 **정확히 1번** 실행되고, 이어서 아무것도 안 눌러도 다시 실행·누적되지 않는다 |
| 직접 입력 우선 | 맡긴 질문과 직접 입력이 동시에 있으면 직접 입력만 실행된다 |
| 지우기 | history 만 비워지고 saved 는 남는다. 예시 버튼이 다시 보이고, 지운 뒤 다음 질문도 정상 실행된다 |

시험은 `service.build_resources` 와 `service.ask` 를 가짜로 바꿨다(`monkeypatch.setattr`). 그래서 **화면 흐름만** 확인하며 답변 품질은 확인하지 않는다.

In [ ]:
# ── 시험의 핵심 도구: AppTest ─────────────────────────────────────────
# AppTest 는 브라우저 없이 Streamlit 스크립트를 실행하고 버튼 클릭·입력을 흉내 낸다.
#
#   at = AppTest.from_file(APP, default_timeout=30)
#   at.run()                                       # 첫 실행
#   [b for b in at.button if b.key == "example-0"][0].click().run()   # 버튼 클릭 → 재실행
#   at.chat_input[0].set_value("질문").run()        # 채팅 입력
#   at.session_state["history"]                    # 세션 상태 읽기
#
# 가짜 ask 가 질문을 리스트에 기록하게 해서 "몇 번, 어떤 질문이 실행됐는지"를 센다.

import subprocess
import sys

# 실행하려면 이 노트북의 셀을 실행하거나, 터미널에서 같은 명령을 쓴다.
# streamlit 이 프로젝트 가상환경에 없어서 uv 로 잠깐 얹어 실행한다(pytest 가 streamlit 을 찾아야 한다).
CMD = ["uv", "run", "--with", "streamlit", "pytest", "tests/rag/ui", "-q"]
print("터미널에서 (프로젝트 루트):", " ".join(CMD))
# subprocess.run(CMD, cwd="C:/mle-02-p2-team2", env={**__import__("os").environ, "PYTHONUTF8": "1", "PYTHONPATH": "src"})

## 6. 직접 해 보기 (연습 문제)
1. `get` 으로 읽기만 하면(`pop` 대신) 어떤 증상이 나올까? `test_example_button_runs_the_question_exactly_once` 가 어떻게 실패하는지 직접 바꿔서 확인해 보자.
2. 예시 질문을 개발자 화면에서만 보이게 하려면 `show_examples()` 호출을 어디에서 어떻게 바꾸면 될까? (힌트: `developer` 변수는 `main()` 에 이미 있다.)
3. 지우기 버튼이 `saved` 도 지우게 바꾸면 어떤 시험이 실패해야 할까? 그 시험을 먼저 쓰고 코드를 바꿔 보자.
4. (생각해 보기) 버튼에 `on_click` 을 쓰지 않고 `if st.button(...)` 으로 만들면 어떤 위험이 있을까? 3번째 절의 "재실행 모델"로 설명해 보자.

## 7. 이번 변경의 한계 (사실만 기록)
- 시험 4개와 기존 UI 시험 17개는 통과했다(AppTest, 가짜 ask). **실제 DB·LLM으로 브라우저에서 눌러 본 확인은 별도 확인이 필요하다.**
- 예시 질문 3개의 답변 상태는 11/08 확인 기록과 같다고 가정할 수 없다(LLM 문장은 실행마다 다를 수 있다).
- 6번 후보(쪽 바로가기)·3번(복사)·4번(피드백)·7번(관련 질문)은 하지 않았다(Codex 검토에서 발표 후로 권고).